# Stage the frozen models for the offline submission

The submission notebook has no internet, so the two frozen models it uses must arrive as an input. This notebook downloads them and saves them under `/kaggle/working/frozen/`:

| Folder | Model | Size |
|---|---|---|
| `frozen/qwen_emb` | [Qwen/Qwen3-Embedding-0.6B](https://huggingface.co/Qwen/Qwen3-Embedding-0.6B) | about 1.2 GB |
| `frozen/bert` | [google-bert/bert-base-cased](https://huggingface.co/google-bert/bert-base-cased) | about 0.4 GB |

**How to run it:** turn **Internet on**, no GPU needed, then **Save Version > Save & Run All**. It takes a few minutes. Afterwards add this notebook's output as an input to the submission notebook.


In [ ]:
import json, shutil, urllib.request
from pathlib import Path

MODELS = {'qwen_emb': 'Qwen/Qwen3-Embedding-0.6B', 'bert': 'google-bert/bert-base-cased'}

def fetch(repo, target):
    """Plain HTTP download with a size check on every file."""
    target = Path(target); target.mkdir(parents=True, exist_ok=True)
    info = json.load(urllib.request.urlopen(f'https://huggingface.co/api/models/{repo}?blobs=true', timeout=60))
    files = {s['rfilename']: s.get('size') for s in info['siblings'] if '/' not in s['rfilename']}
    has_safetensors = any(n.endswith('.safetensors') for n in files)
    for name, size in files.items():
        if not (name.endswith(('.json', '.txt', '.model', '.safetensors')) or (name.endswith('.bin') and not has_safetensors)): continue
        path = target / name
        if path.exists() and path.stat().st_size == size: continue
        with urllib.request.urlopen(f'https://huggingface.co/{repo}/resolve/main/{name}', timeout=120) as source, open(path, 'wb') as sink:
            shutil.copyfileobj(source, sink, 1 << 24)
        assert size is None or path.stat().st_size == size, f'{name} is incomplete'
    return target

for key, repo in MODELS.items():
    folder = fetch(repo, f'/kaggle/working/frozen/{key}')
    weights = [f for f in folder.iterdir() if f.suffix in ('.safetensors', '.bin')]
    assert (folder / 'config.json').exists() and weights, f'{key}: config or weights missing'
    print(f'{key}: {repo} -> {folder} ({sum(f.stat().st_size for f in folder.iterdir()) / 2**30:.2f} GB, weights: {[w.name for w in weights]})')
